In [ ]:
%pip install numpy 
%pip install pandas 
%pip install torch 
%pip install scikit-learn
%pip install tqdm

In [ ]:
import torch

print(torch.__version__)

In [ ]:
import os
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer, StandardScaler
from tqdm import tqdm

# === Paths ===
main_csv = "C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/all_players_ratings_original_updated.csv"
stats_folder = "C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/all_players_fbref_tables"
summary_save_path = "C:/Users/L1160681/OneDrive - TotalEnergies/Documents/Projet/SP/player_stats_summaries"
os.makedirs(summary_save_path, exist_ok=True)

# === Load Data ===
df = pd.read_csv(main_csv)[:500]  # Limit for test
df.dropna(inplace=True)

df["raw_name"] = df["Name"]
df.drop(
    columns=["Unnamed: 0", "Player URL", "Team Link", "fbref_url", "fbref_alltimestat"],
    inplace=True,
)


# === Clean Folder Name ===
def clean_folder_name(name, player_id):
    name_clean = re.sub(
        r"[^\w\s]", "", name
    )  # Remove special characters, preserve spaces
    return f"{name_clean}_{player_id}"


# === Flatten Positions ===
def flatten_positions(pos_string):
    if pd.isnull(pos_string):
        return ["Unknown"]
    output = []
    parts = [p.strip() for p in pos_string.split(",") if p.strip()]
    for part in parts:
        match = re.match(r"^([A-Z]+)\s*\((.*?)\)$", part)
        if match:
            role, locs = match.groups()
            locs = re.findall(r"[A-Z]", locs.upper())
            output.append(role)
            output.extend([loc + role for loc in locs])
        else:
            output.append(part)
    return sorted(set(output)) or ["Unknown"]


df["Flat_Positions"] = df["Positions"].apply(flatten_positions)
df["Primary_Position"] = df["Flat_Positions"].apply(lambda x: x[0] if x else "Unknown")


# === Encode Categorical Features ===
cat_cols = ["Name", "Team", "Nationality", "Primary_Position"]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

# === Multi-hot Encode Position Roles ===
mlb = MultiLabelBinarizer()
multi_pos_df = pd.DataFrame(
    mlb.fit_transform(df["Flat_Positions"]), columns=mlb.classes_, index=df.index
)
df = pd.concat([df, multi_pos_df], axis=1)

# === Scale Age ===
df["Age_Original"] = df["Age"]
scaler_age = StandardScaler()
df["Age"] = scaler_age.fit_transform(df[["Age"]])
df["Age_Scaled"] = df["Age"]

# === Targets ===
target_cols = ["Rating", "Potential", "Value"]
y = df[target_cols].values


# === Load & Save Stats Per Player ===
def load_player_stats(folder_path, player_id, name, save_path):
    folder_name = clean_folder_name(name, player_id)
    full_path = os.path.join(folder_path, folder_name)

    if not os.path.isdir(full_path):
        print(f"Missing folder: {full_path}")
        return None

    summaries = []
    for file in os.listdir(full_path):
        if file.endswith(".csv"):
            file_path = os.path.join(full_path, file)
            try:
                data = pd.read_csv(file_path)
                summary = {}

                numeric = data.select_dtypes(include=np.number)
                for col in numeric.columns:
                    summary[f"{file}_{col}"] = numeric[col].mean()

                for meta_col in ["season", "club", "date", "team", "competition"]:
                    if meta_col in data.columns:
                        summary[f"{file}_{meta_col}"] = data[meta_col].mode().iloc[0]

                summaries.append(pd.Series(summary))
            except Exception as e:
                print(f"Error reading {file}: {e}")

    if summaries:
        final_summary = pd.DataFrame(summaries).mean(numeric_only=True).to_frame().T
        output_file = os.path.join(save_path, f"{player_id}_summary.csv")
        final_summary.to_csv(output_file, index=False)
        return output_file
    return None

In [ ]:
print("Aggregating and saving per-player stats...")

missing_log = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    player_id = row["player_id"]
    player_name = row["raw_name"]

    result = load_player_stats(stats_folder, player_id, player_name, summary_save_path)
    if result is None:
        missing_log.append((player_id, player_name))

print("Done. Player summaries saved to:", summary_save_path)

# === Optional: Save Missing Folder Log ===
if missing_log:
    missing_df = pd.DataFrame(missing_log, columns=["PlayerID", "Name"])
    missing_df.to_csv("missing_folders.csv", index=False)
    print(f"Missing folders logged to missing_folders.csv ({len(missing_log)} entries)")

In [ ]:
# === Load saved summaries and merge ===
summary_files = {
    os.path.splitext(file)[0]: os.path.join(summary_save_path, file)
    for file in os.listdir(summary_save_path)
    if file.endswith(".csv")
}

# Create list of stat dataframes in player order
stats_rows = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    player_id = str(row["player_id"])
    summary_path = summary_files.get(player_id)
    if summary_path:
        try:
            stats = pd.read_csv(summary_path)
            stats_rows.append(stats.iloc[0])  # First row from summary
        except Exception as e:
            print(f"Error loading {summary_path}: {e}")
            stats_rows.append(pd.Series(dtype=float))  # Fallback
    else:
        stats_rows.append(pd.Series(dtype=float))  # Missing summary

stats_df = pd.DataFrame(stats_rows).fillna(0)

# === Combine with original data ===
df_combined = pd.concat(
    [
        df.reset_index(drop=True),
        multi_pos_df.reset_index(drop=True),
        stats_df.reset_index(drop=True),
    ],
    axis=1,
)

df_combined.fillna(0, inplace=True)
# Remove duplicated columns based on name
df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()]
df_combined = df_combined.drop(columns=["D", "F", "M"])

# === Prepare feature matrix ===
X_df = df_combined.drop(columns=target_cols).copy()

# Encode any remaining object columns
for col in X_df.select_dtypes(include=["object"]).columns:
    if col in encoders:
        X_df[col] = encoders[col].transform(X_df[col].astype(str))
    else:
        print(f"Unexpected string column removed: {col}")
        X_df.drop(columns=[col], inplace=True)

# Check again before scaling
object_cols = X_df.select_dtypes(include="object").columns.tolist()
if object_cols:
    print("Dropping unexpected string columns:", object_cols)
    X_df.drop(columns=object_cols, inplace=True)

# Now it's safe to scale
X = X_df.values
# === Final scaling ===
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)


print("Final preprocessing complete. Feature matrix shape:", X_scaled.shape)

In [ ]:
for i in df_combined.columns:
    print(i)

In [ ]:
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Scale targets (y)
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y)
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)


class PlayerDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = PlayerDataset(X_train, y_train)
val_dataset = PlayerDataset(X_val, y_val)
# === Dataloaders ===
train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
    ),
    batch_size=128,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.float32),
    ),
    batch_size=128,
)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=dim, num_heads=heads, batch_first=True
        )
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class ResNetAttentionModel(nn.Module):
    def __init__(self, input_dim, output_dim, projected_dim=256, heads=4, dropout=0.2):
        super().__init__()
        assert projected_dim % heads == 0, (
            "projected_dim must be divisible by number of attention heads"
        )

        # Initial projection
        self.project = nn.Sequential(
            nn.Linear(input_dim, projected_dim), nn.LayerNorm(projected_dim), nn.ReLU()
        )

        # Residual block with skip connection
        self.res_block = nn.Sequential(
            nn.Linear(projected_dim, projected_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(projected_dim, projected_dim),
            nn.LayerNorm(projected_dim),
        )

        # Transformer block for contextual modeling
        self.transformer = TransformerBlock(projected_dim, heads=heads)

        # Attention mechanism
        self.attn = nn.Sequential(
            nn.Linear(projected_dim, 64), nn.Tanh(), nn.Linear(64, 1), nn.Softmax(dim=1)
        )

        # Final prediction head
        self.head = nn.Sequential(
            nn.Linear(projected_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim),
        )

    def forward(self, x):
        x = self.project(x)
        x = self.res_block(x) + x  # residual connection
        x = x.unsqueeze(1)  # add sequence dimension
        x = self.transformer(x)
        weights = self.attn(x)
        x = torch.sum(weights * x, dim=1)  # attention-weighted sum
        return self.head(x)

In [ ]:
# === Model, Optimizer, Criterion ===
model = ResNetAttentionModel(
    input_dim=X.shape[1], output_dim=y.shape[1], projected_dim=256
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-8)
criterion = nn.MSELoss()

# === Training Loop with Early Stopping ===
best_val_loss = float("inf")
patience = 10
counter = 0

for epoch in range(1, 201):
    model.train()
    train_loss = 0
    train_loop = tqdm(train_loader, desc=f"[Epoch {epoch}] Train", leave=False)
    for batch_x, batch_y in train_loop:
        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_loop.set_postfix(batch_loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f" Epoch {epoch:03d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}"
    )

    # === Early Stopping Logic ===
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break


In [ ]:
# === Get true validation targets (unscaled) ===
y_val_unscaled = scaler_y.inverse_transform(y_val)

# === Run model and inverse-transform predictions ===
model.eval()
all_preds = []
with torch.no_grad():
    for batch_x, _ in val_loader:
        preds = model(batch_x)
        all_preds.append(preds.numpy())

# === Concatenate predictions and inverse scale ===
scaled_preds = np.vstack(all_preds)
unscaled_preds = scaler_y.inverse_transform(scaled_preds)

# === Display predicted vs actual values ===
print("Showing predicted vs actual values for validation set:\n")
for i in range(1):  # First 30 examples
    pred_rating, pred_potential, pred_value = unscaled_preds[i]
    true_rating, true_potential, true_value = y_val_unscaled[i]
    print(
        f"Player {i + 1}: "
        f"Predicted → Rating: {pred_rating:.2f}, Potential: {pred_potential:.2f}, Value: €{pred_value:,.0f} | "
        f"Actual → Rating: {true_rating:.2f}, Potential: {true_potential:.2f}, Value: €{true_value:,.0f}"
    )

In [ ]:
class EnhancedTransformerResNet(nn.Module):
    def __init__(
        self, input_dim, output_dim, hidden_dim=256, depth=2, heads=4, dropout=0.3
    ):
        super().__init__()
        assert hidden_dim % heads == 0

        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.LayerNorm(hidden_dim)
        )

        # Stack multiple Transformer blocks
        self.transformer_blocks = nn.ModuleList(
            [TransformerBlock(hidden_dim, heads=heads) for _ in range(depth)]
        )

        # Gated residual MLP block
        self.res_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # Attention weights for global pooling
        self.attn_weights = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.Tanh(), nn.Linear(64, 1), nn.Softmax(dim=1)
        )

        # Output head
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = x.unsqueeze(1)  # Add sequence dimension

        for block in self.transformer_blocks:
            x = block(x)

        # Optional gated residual block
        res = self.res_mlp(x)
        x = x + res

        # Attention pooling
        weights = self.attn_weights(x)
        pooled = torch.sum(weights * x, dim=1)

        return self.head(pooled)


# === Model Definitions ===
class BottleneckFCBlock(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout_rate=0.2, residual_scale=0.8):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.activation1 = nn.SiLU()
        self.dropout1 = nn.Dropout(dropout_rate)

        self.fc2 = nn.Linear(hidden_dim, in_dim)
        self.norm2 = nn.LayerNorm(in_dim)
        self.activation2 = nn.SiLU()
        self.dropout2 = nn.Dropout(dropout_rate)

        self.residual_scale = residual_scale

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.activation1(self.norm1(out))
        out = self.dropout1(out)

        out = self.fc2(out)
        out = self.activation2(self.norm2(out))
        out = self.dropout2(out)

        return identity + self.residual_scale * out


class PositionEmbedder(nn.Module):
    def __init__(self, num_positions, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_positions, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, position_indices):
        """
        position_indices: LongTensor of shape [batch_size, num_positions_per_player]
        Returns: pooled embedding of shape [batch_size, embed_dim]
        """
        pos_embed = self.embedding(position_indices)  # [B, P, D]
        pooled = pos_embed.mean(dim=1)  # average across positions
        return self.norm(pooled)


class RatingPotentialModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.init_proj = nn.Sequential(
            nn.Linear(input_dim, 256), nn.GELU(), nn.Dropout(0.2)
        )

        self.mixer = nn.Sequential(
            BottleneckFCBlock(256, 128),
            BottleneckFCBlock(256, 192),
            BottleneckFCBlock(256, 64),
        )

        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.SiLU(), nn.Dropout(0.2), nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.init_proj(x)
        x = self.mixer(x)
        return self.head(x)


class ValueFromRPModel(nn.Module):
    def __init__(self, input_dim, embed_dim=10, num_positions=4):
        super().__init__()
        self.position_embedder = PositionEmbedder(num_positions, embed_dim)
        concat_dim = input_dim + embed_dim

        self.encoder = nn.Sequential(
            nn.Linear(concat_dim, 192), nn.ReLU(), nn.Dropout(0.2)
        )

        self.processing_blocks = nn.Sequential(
            BottleneckFCBlock(192, 96),
            BottleneckFCBlock(192, 128),
            BottleneckFCBlock(192, 64),
        )

        self.output_layer = nn.Sequential(
            nn.Linear(192, 64), nn.GELU(), nn.Linear(64, 1)
        )

    def forward(self, x_numeric, position_indices):
        """
        position_indices: LongTensor of shape [batch_size, num_positions_per_player]
        """
        pos_embed = self.position_embedder(position_indices)  # [B, embed_dim]
        x = torch.cat([x_numeric, pos_embed], dim=1)
        x = self.encoder(x)
        x = self.processing_blocks(x)
        return self.output_layer(x)


class RatingPotentialPerPosition(nn.Module):
    def __init__(self, input_dim, num_positions, embed_dim=12):
        super().__init__()
        self.position_embedder = PositionEmbedder(num_positions, embed_dim)
        self.feature_proj = nn.Sequential(
            nn.Linear(input_dim + embed_dim, 256), nn.GELU(), nn.Dropout(0.2)
        )
        self.mixer = nn.Sequential(
            BottleneckFCBlock(256, 128),
            BottleneckFCBlock(256, 192),
        )
        self.head = nn.Sequential(
            nn.Linear(256, 2)  # Rating, Potential
        )

    def forward(self, features, position_indices):
        pos_embeds = self.position_embedder(position_indices)
        x = torch.cat([features, pos_embeds], dim=1)
        x = self.feature_proj(x)
        x = self.mixer(x)
        return self.head(x)


def select_best_position_prediction(model, features_list, position_indices_list):
    all_preds = []
    for features, pos_idx in zip(features_list, position_indices_list):
        pred = model(features.unsqueeze(0), pos_idx.unsqueeze(0))
        all_preds.append(pred)
    all_preds = torch.stack(all_preds).squeeze(1)  # shape [num_positions, 2]
    best_index = torch.argmax(all_preds[:, 1])  # max potential
    return all_preds[best_index], best_index

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, TensorDataset

# === Targets ===
y_rating_potential = y[:, :2]  # Rating, Potential
y_value = y[:, 2:]  # Value only

# === Remove duplicated columns from X_df ===
X_df = pd.DataFrame(
    X, columns=[f"feat_{i}" for i in range(X.shape[1])]
)  # optional naming if needed
X_df = X_df.loc[:, ~X_df.columns.duplicated()]
X = X_df.values

# === Feature Scaling ===
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y_rp = StandardScaler()
y_rp_scaled = scaler_y_rp.fit_transform(y_rating_potential)

scaler_y_value = StandardScaler()
y_value_scaled = scaler_y_value.fit_transform(y_value)

# === Position Features (multi-hot or label-based) ===
# You can swap this with Primary_Position if preferred
pos = multi_pos_df.values

# === Train/Test Split ===
X_train, X_val, y_rp_train, y_rp_val, y_value_train, y_value_val, pos_train, pos_val = (
    train_test_split(
        X_scaled, y_rp_scaled, y_value_scaled, pos, test_size=0.2, random_state=42
    )
)

# === DataLoaders: RP model ===
rp_train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_rp_train, dtype=torch.float32),
    ),
    batch_size=128,
    shuffle=True,
)

rp_val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_rp_val, dtype=torch.float32),
    ),
    batch_size=128,
    shuffle=False,
)

# === DataLoaders: Value model ===
value_train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(pos_train, dtype=torch.float32),
        torch.tensor(y_value_train, dtype=torch.float32),
    ),
    batch_size=128,
    shuffle=True,
)

value_val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(pos_val, dtype=torch.float32),
        torch.tensor(y_value_val, dtype=torch.float32),
    ),
    batch_size=128,
    shuffle=False,
)


# === Instantiate Models ===
rp_model = RatingPotentialModel(input_dim=X_train.shape[1])
print(f"Initializing RP Model with input_dim = {X.shape[1]}")
value_model = ValueFromRPModel(
    input_dim=X.shape[1] + pos.shape[1]
)  # If concatenating X + pos

# === Optimizers & Schedulers ===
optimizer_rp = torch.optim.Adam(rp_model.parameters(), lr=1e-3)
optimizer_value = torch.optim.Adam(value_model.parameters(), lr=1e-3)

scheduler_rp = StepLR(optimizer_rp, step_size=10, gamma=0.1)
scheduler_value = StepLR(optimizer_value, step_size=10, gamma=0.1)

# === Loss functions ===
criterion_rp = nn.HuberLoss()
criterion_value = nn.HuberLoss()


# === Model Saving Utility ===
def save_models():
    torch.save(rp_model.state_dict(), "rp_model_best.pt")
    torch.save(value_model.state_dict(), "value_model_best.pt")


print("Pipeline ready: scaled, split, loaders built, models defined, optimizers armed.")

In [ ]:
best_loss = float("inf")
patience = 10
counter = 0

rp_model.train()
epoch = 0

while True:
    total_loss = 0
    for batch_x, batch_y in rp_train_loader:
        optimizer_rp.zero_grad()
        preds = rp_model(batch_x)
        loss = criterion_rp(preds, batch_y)
        loss.backward()
        optimizer_rp.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(rp_train_loader)
    print(f"[RP Epoch {epoch + 1}] Loss: {avg_loss:.4f}")

    # Early stopping check
    if avg_loss < best_loss:
        best_loss = avg_loss
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break

    epoch += 1

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# === Step 0: Position Indexing (with 'PAD' token) ===
all_unique_positions = sorted(
    set(pos for pos_list in df["Flat_Positions"] for pos in pos_list)
)
all_positions = ["PAD"] + all_unique_positions
pos_map = {pos: idx for idx, pos in enumerate(all_positions)}
pad_idx = pos_map["PAD"]

max_pos_len = max(len(pos_list) for pos_list in df["Flat_Positions"])
position_idx_list = [
    [pos_map[pos] for pos in pos_list] + [pad_idx] * (max_pos_len - len(pos_list))
    for pos_list in df["Flat_Positions"]
]
position_tensor = torch.tensor(position_idx_list, dtype=torch.long)

# === Step 1: Define feature columns (excluding targets and raw text) ===
excluded_cols = ["Rating", "Potential", "Value", "Flat_Positions"]
selected_features = [
    col
    for col in df.columns
    if col not in excluded_cols and df[col].dtype in [np.float64, np.int64]
]

X_raw = df[selected_features].copy().values
y_raw = df[["Rating", "Potential", "Value"]].copy().values

# === Step 2: Feature Scaling ===
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X_raw)

# === Step 3: Target Scaling (optional but recommended for regression)
scaler_y_rp = StandardScaler()
scaler_y_value = StandardScaler()

y_rp_scaled = scaler_y_rp.fit_transform(y_raw[:, :2])  # Rating + Potential
y_value_scaled = scaler_y_value.fit_transform(y_raw[:, 2:3])  # Value (column vector)

# === Step 4: Train/Test Split (sync across all tensors)
train_idx, val_idx = train_test_split(
    np.arange(len(X_scaled)), test_size=0.2, random_state=42
)

# Split inputs and targets
X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
y_rp_train, y_rp_val = y_rp_scaled[train_idx], y_rp_scaled[val_idx]
y_value_train, y_value_val = y_value_scaled[train_idx], y_value_scaled[val_idx]
pos_train, pos_val = position_tensor[train_idx], position_tensor[val_idx]

In [ ]:
# === Step 4: Predict RP outputs ===
rp_model.eval()
with torch.no_grad():
    rp_preds_train = rp_model(torch.tensor(X_train, dtype=torch.float32)).numpy()
    rp_preds_val = rp_model(torch.tensor(X_val, dtype=torch.float32)).numpy()

# === Step 5: Retrieve Scaled Age ===
age_train = df["Age"].values[train_idx].reshape(-1, 1)
age_val = df["Age"].values[val_idx].reshape(-1, 1)

# === Step 6: Stack RP + Age
value_input_train = np.hstack([rp_preds_train, age_train])
value_input_val = np.hstack([rp_preds_val, age_val])

# === Step 7: TensorDataset (Safe Casting) ===

# Convert input arrays to tensors with appropriate types
value_input_train_tensor = torch.tensor(value_input_train, dtype=torch.float32)
value_input_val_tensor = torch.tensor(value_input_val, dtype=torch.float32)
pos_train_tensor = torch.tensor(pos_train, dtype=torch.long)
pos_val_tensor = torch.tensor(pos_val, dtype=torch.long)
y_value_train_tensor = torch.tensor(y_value_train.reshape(-1, 1), dtype=torch.float32)
y_value_val_tensor = torch.tensor(y_value_val.reshape(-1, 1), dtype=torch.float32)

# Create TensorDataset for training and validation
train_dataset = TensorDataset(
    value_input_train_tensor, pos_train_tensor, y_value_train_tensor
)

val_dataset = TensorDataset(value_input_val_tensor, pos_val_tensor, y_value_val_tensor)

# Wrap with DataLoader for batching
value_train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
value_val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

# === Step 8: Initialize Value Model ===
value_model = ValueFromRPModel(
    input_dim=3,  # RP output (2) + Age (1)
    embed_dim=0,
    num_positions=len(pos_map),
)


In [ ]:
# === Step 9: Training ===
optimizer_value = torch.optim.Adam(value_model.parameters(), lr=1e-3)
criterion_value = nn.MSELoss()

best_val_loss = float("inf")
patience = 10
patience_counter = 0
max_epochs = 10000

for epoch in range(max_epochs):
    value_model.train()
    total_loss = 0.0

    for batch_x, batch_pos, batch_y in value_train_loader:
        try:
            optimizer_value.zero_grad()

            # Shape check before forward
            assert batch_x.shape[1] == value_model.encoder[0].in_features, (
                f"batch_x shape mismatch: expected {value_model.encoder[0].in_features}, got {batch_x.shape[1]}"
            )
            assert batch_pos.shape[1] == pos_train.shape[1], (
                f"batch_pos shape mismatch: expected {pos_train.shape[1]}, got {batch_pos.shape[1]}"
            )

            preds = value_model(batch_x, batch_pos)
            loss = criterion_value(preds, batch_y)

            loss.backward()
            optimizer_value.step()
            total_loss += loss.item()

        except Exception as e:
            print(f"Training Error at epoch {epoch + 1}: {e}")
            print(f"batch_x shape: {batch_x.shape}")
            print(f"batch_pos shape: {batch_pos.shape}")
            print(f"batch_y shape: {batch_y.shape}")
            break  # Optional: halt on error

    avg_train_loss = total_loss / len(value_train_loader)

    # Validation loop
    value_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_pos, batch_y in value_val_loader:
            try:
                preds = value_model(batch_x, batch_pos)
                loss = criterion_value(preds, batch_y)
                val_loss += loss.item()
            except Exception as e:
                print(f"Validation Error at epoch {epoch + 1}: {e}")
                break

    avg_val_loss = val_loss / len(value_val_loader)

    print(
        f"[Epoch {epoch + 1}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}"
    )

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(value_model.state_dict(), "best_value_model.pt")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(
                f" Early stopping at epoch {epoch + 1}. Best Val Loss: {best_val_loss:.4f}"
            )
            break

In [ ]:
import numpy as np
import pandas as pd
import torch

# === 1. Set models to eval mode ===
rp_model.eval()
value_model.eval()

# === 2. Recover metadata using validation indices ===
player_names = df_combined.loc[val_idx, "raw_name"].values
age_val_unscaled = df_combined.loc[val_idx, "Age_Original"].values

# === 3. Scale age ===
age_scaled_val = scaler_age.transform(age_val_unscaled.reshape(-1, 1))

# === 4. Map position roles ===
# Replace label decoding with full role list from Flat_Positions
flat_roles = (
    df_combined.loc[val_idx, "Flat_Positions"]
    .apply(lambda roles: ", ".join(roles))
    .values
)

# === 5. Predict RP (Rating & Potential) and unscale ===
rp_preds_scaled = rp_model(torch.tensor(X_val, dtype=torch.float32)).detach().numpy()
rp_preds_unscaled = scaler_y_rp.inverse_transform(rp_preds_scaled)

# === 6. Prepare input for value prediction ===
value_inputs = np.hstack([rp_preds_scaled, age_scaled_val])
value_inputs_tensor = torch.tensor(value_inputs, dtype=torch.float32)

# If your value_model requires positional IDs, ensure pos_val_tensor is correct
pos_val_int = X_df.loc[val_idx, "Flat_Positions"].astype(int).values
pos_val_tensor = torch.tensor(pos_val_int, dtype=torch.long)

# === 7. Predict value and unscale ===
value_preds_scaled = value_model(value_inputs_tensor, pos_val_tensor).detach().numpy()
value_preds_unscaled = scaler_y_value.inverse_transform(value_preds_scaled)

# === 8. Unscale ground-truth RP and Value ===
y_rp_val_unscaled = scaler_y_rp.inverse_transform(y_rp_val)
y_value_val_unscaled = scaler_y_value.inverse_transform(y_value_val)

# === 9. Display output ===
print(" Player Predictions vs Actuals (All Positions):")
for i in range(min(100, len(X_val))):
    name = player_names[i]
    age = age_val_unscaled[i]
    positions = flat_roles[i]

    pred_rating, pred_potential = rp_preds_unscaled[i]
    pred_value = value_preds_unscaled[i][0]

    true_rating, true_potential = y_rp_val_unscaled[i]
    true_value = y_value_val_unscaled[i][0]

    print(
        f"{name:25} | Age: {age:.1f} | Positions: {positions:30} | "
        f"Pred → Rating: {pred_rating:.2f}, Potential: {pred_potential:.2f}, Value: €{pred_value:,.0f} | "
        f"Actual → Rating: {true_rating:.2f}, Potential: {true_potential:.2f}, Value: €{true_value:,.0f}"
    )

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
def build_position_vocab(df):
    all_positions = sorted(
        set(pos for positions in df["Flat_Positions"] for pos in positions)
    )
    return {pos: idx for idx, pos in enumerate(all_positions)}


def expand_player_positions(df, numeric_cols, position_vocab):
    rows = []
    for _, row in df.iterrows():
        for pos in row["Flat_Positions"]:
            new_row = {col: row[col] for col in numeric_cols}
            new_row["player_id"] = row["player_id"]
            new_row["raw_name"] = row["raw_name"]
            new_row["position"] = pos
            new_row["position_index"] = position_vocab[pos]
            new_row["rating"] = row["Rating"]
            new_row["potential"] = row["Potential"]
            rows.append(new_row)
    return pd.DataFrame(rows)

In [ ]:
class BottleneckFCBlock(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout_rate=0.4, residual_scale=0.8):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.activation1 = nn.SiLU()
        self.dropout1 = nn.Dropout(dropout_rate)

        self.fc2 = nn.Linear(hidden_dim, in_dim)
        self.norm2 = nn.LayerNorm(in_dim)
        self.activation2 = nn.SiLU()
        self.dropout2 = nn.Dropout(dropout_rate)
        self.residual_scale = residual_scale

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.activation1(self.norm1(out))
        out = self.dropout1(out)
        out = self.fc2(out)
        out = self.activation2(self.norm2(out))
        out = self.dropout2(out)
        return identity + self.residual_scale * out


class PositionEmbedder(nn.Module):
    def __init__(self, num_positions, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_positions, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, position_indices):
        pos_embed = self.embedding(position_indices)  # [B, P, D]
        pooled = pos_embed.mean(dim=1)  # average across positions
        return self.norm(pooled)


class RatingPotentialPerPosition(nn.Module):
    def __init__(
        self, input_dim, num_positions, hidden_dim=64
    ):  # Add hidden_dim if needed
        super().__init__()
        self.pos_embedding = nn.Embedding(num_positions, 8)
        self.fc1 = nn.Linear(input_dim + 8, hidden_dim)
        self.relu = nn.ReLU()
        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dim, 2),
            nn.Identity(),  # No squashing — predictions stay in [0, 1] and will be unscaled later
        )

    def forward(self, x_num, x_pos):
        pos_emb = self.pos_embedding(x_pos)
        x = torch.cat([x_num, pos_emb], dim=1)
        x = self.relu(self.fc1(x))
        return self.output_layer(x)

In [ ]:
numeric_cols = [
    "AM",
    "CAM",
    "CD",
    "CDM",
    "CF",
    "CM",
    "DM",
    "LAM",
    "LD",
    "LF",
    "LM",
    "RAM",
    "RD",
    "RF",
    "RM",
]

position_vocab = build_position_vocab(df)
expanded_df = expand_player_positions(df, numeric_cols, position_vocab)

X_numeric = expanded_df[numeric_cols].values
X_position = expanded_df["position_index"].values
y = expanded_df[["rating", "potential"]].values

X_num_train, X_num_val, X_pos_train, X_pos_val, y_train, y_val = train_test_split(
    X_numeric, X_position, y, test_size=0.4, random_state=42
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_rp = MinMaxScaler(feature_range=(0, 100))
scaler_rp.fit(y_train)
y_train_scaled = scaler_rp.transform(y_train)
y_val_scaled = scaler_rp.transform(y_val)


In [ ]:
class PlayerPositionDataset(torch.utils.data.Dataset):
    def __init__(self, X_num, X_pos, y):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_pos = torch.tensor(X_pos, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_num[idx], self.X_pos[idx], self.y[idx]


train_dataset = PlayerPositionDataset(X_num_train, X_pos_train, y_train)
val_dataset = PlayerPositionDataset(X_num_val, X_pos_val, y_val)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RatingPotentialPerPosition(
    input_dim=len(numeric_cols), num_positions=len(position_vocab)
).to(device)
criterion = nn.HuberLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)


def train_model(
    model, train_loader, val_loader, max_epochs=500, patience=20, min_delta=1e-3
):
    best_val_loss = float("inf")
    best_epoch = 0
    wait = 0
    epoch = 0
    history = []

    while epoch < max_epochs:
        epoch += 1
        model.train()
        total_loss = 0

        for X_num, X_pos, y in train_loader:
            X_num, X_pos, y = X_num.to(device), X_pos.to(device), y.to(device)
            preds = model(X_num, X_pos)
            loss = criterion(preds, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)

        # Validation
        model.eval()
        with torch.no_grad():
            val_loss = sum(
                criterion(model(Xn.to(device), Xp.to(device)), yb.to(device)).item()
                for Xn, Xp, yb in val_loader
            ) / len(val_loader)

        history.append((epoch, avg_train_loss, val_loss))
        print(
            f"Epoch {epoch}: Train Loss={avg_train_loss:.4f}, Val Loss={val_loss:.4f}"
        )

        # Early stopping logic
        if val_loss + min_delta < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            wait = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            wait += 1
            if wait >= patience:
                print(
                    f"Early stopping triggered at epoch {epoch}. Best epoch: {best_epoch}"
                )
                break

    return history

In [ ]:
def evaluate_best_position(model, player_numeric_data, position_indices):
    model.eval()
    X_num = torch.tensor(player_numeric_data, dtype=torch.float32).to(device)
    X_pos = torch.tensor(position_indices, dtype=torch.long).to(device)
    with torch.no_grad():
        preds = model(X_num, X_pos)  # [num_positions, 2]
        best_idx = torch.argmax(preds[:, 1])
        return preds[best_idx].cpu().numpy(), best_idx.item()

In [ ]:
train_model(model, train_loader, val_loader)

In [ ]:
def evaluate_players_per_position(df, model, numeric_cols, position_vocab, scaler_rp):
    model.eval()
    predictions = []

    for _, row in df.iterrows():
        exclude_positions = {"M", "F", "D"}  # Positions to ignore

        positions = [p for p in row["Flat_Positions"] if p not in exclude_positions]

        if not positions:
            continue

        player_features = np.array([row[col] for col in numeric_cols])
        features_expanded = np.tile(player_features, (len(positions), 1))
        position_indices = [position_vocab[pos] for pos in positions]

        X_num_tensor = torch.tensor(features_expanded, dtype=torch.float32)
        X_pos_tensor = torch.tensor(position_indices, dtype=torch.long)

        with torch.no_grad():
            preds_scaled = model(X_num_tensor, X_pos_tensor).cpu().numpy()
            preds_unscaled = scaler_rp.inverse_transform(preds_scaled)

        best_idx = np.argmax(preds_unscaled[:, 1])  # Best potential
        best_rating, best_potential = preds_unscaled[best_idx]
        best_position = positions[best_idx]

        predictions.append(
            {
                "player_id": row["player_id"],
                "raw_name": row["raw_name"],
                "age": row["Age_Original"],
                "best_position": best_position,
                "pred_rating": best_rating,
                "pred_potential": best_potential,
            }
        )

    return pd.DataFrame(predictions)

In [ ]:
pred_df = evaluate_players_per_position(
    df, model, numeric_cols, position_vocab, scaler_rp
)

print("\n🔝 Best Position-Based Predictions:\n")
for _, row in pred_df.iterrows():
    print(
        f"{row['raw_name']:25} | Age: {row['age']:.1f} | Position: {row['best_position']:4} | "
        f"Rating: {row['pred_rating']:.2f} | Potential: {row['pred_potential']:.2f}"
    )

In [ ]:
def debug_single_player(row, model, numeric_cols, position_vocab, scaler_rp):
    model.eval()

    positions = row["Flat_Positions"]
    player_features = np.array([row[col] for col in numeric_cols])
    features_expanded = np.tile(player_features, (len(positions), 1))
    pos_indices = [position_vocab[p] for p in positions]

    X_num = torch.tensor(features_expanded, dtype=torch.float32)
    X_pos = torch.tensor(pos_indices, dtype=torch.long)

    with torch.no_grad():
        preds_scaled = model(X_num, X_pos).cpu().numpy()
        preds_unscaled = scaler_rp.inverse_transform(preds_scaled)
        print("Scaled predictions:", preds_scaled[:5])
        print("Unscaled predictions:", scaler_rp.inverse_transform(preds_scaled[:5]))

    # Print for inspection
    for pos, scaled, unscaled in zip(positions, preds_scaled, preds_unscaled):
        print(f"Position: {pos:4} | Scaled: {scaled} | Unscaled: {unscaled}")

In [ ]:
player_row = df[df["raw_name"] == "Kylian Mbappé"].iloc[0]
debug_single_player(player_row, model, numeric_cols, position_vocab, scaler_rp)

In [ ]:
def print_scaled_predictions(row, model, numeric_cols, position_vocab):
    model.eval()

    positions = row["Flat_Positions"]
    player_features = np.array([row[col] for col in numeric_cols])
    features_expanded = np.tile(player_features, (len(positions), 1))
    pos_indices = [position_vocab[p] for p in positions]

    X_num = torch.tensor(features_expanded, dtype=torch.float32)
    X_pos = torch.tensor(pos_indices, dtype=torch.long)

    with torch.no_grad():
        preds_scaled = model(X_num, X_pos).cpu().numpy()

    # Print the raw scaled predictions
    print(f"\n🔬 Scaled predictions for {row['raw_name']}:\n")
    for pos, scaled in zip(positions, preds_scaled):
        rating_scaled, potential_scaled = scaled
        print(
            f"Position: {pos:4} | Scaled Rating: {rating_scaled:.4f} | Scaled Potential: {potential_scaled:.4f}"
        )

In [ ]:
def print_unscaled_predictions(row, model, numeric_cols, position_vocab, scaler_rp):
    model.eval()

    positions = row["Flat_Positions"]
    player_features = np.array([row[col] for col in numeric_cols])
    features_expanded = np.tile(player_features, (len(positions), 1))
    pos_indices = [position_vocab[p] for p in positions]

    X_num = torch.tensor(features_expanded, dtype=torch.float32)
    X_pos = torch.tensor(pos_indices, dtype=torch.long)

    with torch.no_grad():
        preds_scaled = model(X_num, X_pos).cpu().numpy()

        normalized_preds = (
            preds_scaled / 100.0 if preds_scaled.max() > 1.5 else preds_scaled
        )
        preds_unscaled = scaler_rp.inverse_transform(normalized_preds)

    print(f"\n🎯 Unscaled predictions for {row['raw_name']}:\n")
    for pos, unscaled in zip(positions, preds_unscaled):
        rating, potential = unscaled
        print(f"Position: {pos:4} | Rating: {rating:.1f} | Potential: {potential:.1f}")

In [ ]:
def evaluate_scaled_predictions(df, model, numeric_cols, position_vocab):
    model.eval()
    scaled_preds = []

    for _, row in df.iterrows():
        exclude_positions = {"M", "F", "D"}  # Positions to ignore

        positions = [p for p in row["Flat_Positions"] if p not in exclude_positions]
        if not positions:
            continue

        player_features = np.array([row[col] for col in numeric_cols])
        features_expanded = np.tile(player_features, (len(positions), 1))
        position_indices = [position_vocab[pos] for pos in positions]

        X_num_tensor = torch.tensor(features_expanded, dtype=torch.float32)
        X_pos_tensor = torch.tensor(position_indices, dtype=torch.long)

        with torch.no_grad():
            preds_scaled = model(X_num_tensor, X_pos_tensor).cpu().numpy()

        best_idx = np.argmax(preds_scaled[:, 1])  # Max potential
        rating_scaled, potential_scaled = preds_scaled[best_idx]
        best_position = positions[best_idx]

        scaled_preds.append(
            {
                "player_id": row["player_id"],
                "raw_name": row["raw_name"],
                "age": row["Age_Original"],
                "best_position": best_position,
                "scaled_rating": rating_scaled,
                "scaled_potential": potential_scaled,
            }
        )

    return pd.DataFrame(scaled_preds)

In [ ]:
def evaluate_unscaled_predictions(df, model, numeric_cols, position_vocab, scaler_rp):
    model.eval()
    unscaled_preds = []

    for _, row in df.iterrows():
        exclude_positions = {"M", "F", "D"}  # Positions to ignore
        positions = [p for p in row["Flat_Positions"] if p not in exclude_positions]
        if not positions:
            continue

        # Prepare inputs
        player_features = np.array([row[col] for col in numeric_cols])
        features_expanded = np.tile(player_features, (len(positions), 1))
        position_indices = [position_vocab[pos] for pos in positions]

        X_num_tensor = torch.tensor(features_expanded, dtype=torch.float32)
        X_pos_tensor = torch.tensor(position_indices, dtype=torch.long)

        with torch.no_grad():
            preds_scaled = model(X_num_tensor, X_pos_tensor).cpu().numpy()

            normalized_preds = (
                preds_scaled / 100.0 if preds_scaled.max() > 1.5 else preds_scaled
            )
            preds_unscaled = scaler_rp.inverse_transform(normalized_preds)

        best_idx = np.argmax(preds_unscaled[:, 1])  # Max potential
        rating_unscaled, potential_unscaled = preds_unscaled[best_idx]
        best_position = positions[best_idx]

        unscaled_preds.append(
            {
                "player_id": row["player_id"],
                "raw_name": row["raw_name"],
                "age": row["Age_Original"],
                "best_position": best_position,
                "rating": rating_unscaled,
                "potential": potential_unscaled,
            }
        )

    return pd.DataFrame(unscaled_preds)

In [ ]:
scaled_df = evaluate_scaled_predictions(df, model, numeric_cols, position_vocab)

print("Scaled Position-Based Predictions:")
for _, row in scaled_df.iterrows():
    print(
        f"{row['raw_name']:25} | Age: {row['age']:.1f} | Position: {row['best_position']:4} | "
        f"Scaled Rating: {row['scaled_rating']:.4f} | Scaled Potential: {row['scaled_potential']:.4f}"
    )

In [ ]:
final_df = evaluate_unscaled_predictions(
    df, model, numeric_cols, position_vocab, scaler_rp
)

print("Unscaled Position-Based Predictions:")
for _, row in final_df.iterrows():
    print(
        f"{row['raw_name']:25} | Age: {row['age']:.1f} | Position: {row['best_position']:4} | "
        f"Rating: {row['rating']:.1f} | Potential: {row['potential']:.1f}"
    )

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models


class GeneralRatingModel(nn.Module):
    def __init__(self, hidden_dim=64, pretrained=True):
        super().__init__()

        # Load pretrained ResNet50 and remove its classifier head
        resnet = models.resnet50(pretrained=pretrained)
        self.feature_extractor = nn.Sequential(
            *list(resnet.children())[:-1]
        )  # Remove final FC layer

        # Freeze layers if you don't want them to be fine-tuned
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        # New classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),  # ResNet output shape is (batch, 2048, 1, 1)
            nn.Linear(2048, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 2),  # Output: rating, potential
        )

    def forward(self, x_img):
        features = self.feature_extractor(x_img)
        return self.classifier(features)

In [ ]:
def train_general_model(
    model, train_loader, val_loader, max_epochs=500, patience=20, min_delta=1e-4
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.HuberLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    best_val_loss = float("inf")
    wait = 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0

        for X_num, y in train_loader:  # No position input
            X_num, y = X_num.to(device), y.to(device)
            preds = model(X_num)
            loss = criterion(preds, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)

        # Validation
        model.eval()
        with torch.no_grad():
            val_loss = sum(
                criterion(model(Xn.to(device)), yb.to(device)).item()
                for Xn, yb in val_loader
            ) / len(val_loader)

        print(
            f"Epoch {epoch}: Train Loss={avg_train_loss:.4f}, Val Loss={val_loss:.4f}"
        )

        if val_loss + min_delta < best_val_loss:
            best_val_loss = val_loss
            wait = 0
            torch.save(model.state_dict(), "best_general_model.pt")
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping triggered. Best val loss: {best_val_loss:.4f}")
                break

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_general_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_num_train, dtype=torch.float32),
        torch.tensor(y_train_scaled, dtype=torch.float32),
    ),
    batch_size=64,
    shuffle=True,
)

val_general_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_num_val, dtype=torch.float32),
        torch.tensor(y_val_scaled, dtype=torch.float32),
    ),
    batch_size=64,
)

general_model = GeneralRatingModel(input_dim=len(numeric_cols)).to(device)

# Then train it
train_general_model(general_model, train_general_loader, val_general_loader)

In [ ]:
def evaluate_blended_predictions(
    df, per_pos_model, general_model, numeric_cols, position_vocab, scaler_rp
):
    per_pos_model.eval()
    general_model.eval()
    predictions = []

    for _, row in df.iterrows():
        exclude_positions = {"M", "F", "D"}
        positions = [p for p in row["Flat_Positions"] if p not in exclude_positions]
        if not positions:
            continue

        player_features = np.array([row[col] for col in numeric_cols])
        features_expanded = np.tile(player_features, (len(positions), 1))
        position_indices = [position_vocab[pos] for pos in positions]

        X_num_tensor = torch.tensor(features_expanded, dtype=torch.float32)
        X_pos_tensor = torch.tensor(position_indices, dtype=torch.long)
        X_num_general = torch.tensor(player_features, dtype=torch.float32).unsqueeze(0)

        with torch.no_grad():
            per_pos_scaled = per_pos_model(X_num_tensor, X_pos_tensor).cpu().numpy()
            general_scaled = general_model(X_num_general).cpu().numpy()
            general_repeated = np.repeat(general_scaled, len(positions), axis=0)

            blended_scaled = (3 * per_pos_scaled + general_repeated) / 4.0
            blended_unscaled = scaler_rp.inverse_transform(blended_scaled)

        best_idx = np.argmax(blended_unscaled[:, 1])  # Based on potential
        best_position = positions[best_idx]
        best_rating, best_potential = blended_unscaled[best_idx]

        predictions.append(
            {
                "player_id": row["player_id"],
                "raw_name": row["raw_name"],
                "age": row["Age_Original"],
                "best_position": best_position,
                "blended_rating": best_rating,
                "blended_potential": best_potential,
            }
        )

    return pd.DataFrame(predictions)

In [ ]:
scaled_df = evaluate_scaled_predictions(df, model, numeric_cols, position_vocab)

print("Scaled Position-Based Predictions:")
for _, row in scaled_df.iterrows():
    print(
        f"{row['raw_name']:25} | Age: {row['age']:.1f} | Position: {row['best_position']:4} | "
        f"Scaled Rating: {row['scaled_rating']:.4f} | Scaled Potential: {row['scaled_potential']:.4f}"
    )